# In-Context KL Divergence & $k$-HMM Baseline Analysis

Adapted from **Poncin et al. (2026)**, *Large Language Models Develop Belief State Geometry In-Context* ([arXiv:2609.17376v1](https://arxiv.org/abs/2609.17376)):

1. **Per-Position In-Context KL Divergence (Figure 2, left)**:
   - Computes $D_{\mathrm{KL}}(P_{\mathrm{true}} \parallel Q)$ across usable token positions in long trajectories ($m = 50\text{--}200$ ticks).
   - Compared against analytic baselines: **0-HMM** (stationary/marginal distribution) and **1-HMM** (conditioned on single preceding letter).
   - Selectable **linear** and **log** scales with 95% confidence interval error bands.
   - **Decomposition Panel**: Decomposes total model KL into in-subspace belief distribution error and off-branch physics mass leakage ($-\ln Z$).

2. **Crossover Markov Order $k^*$ (Figure 2, right)**:
   - Evaluates average converged KL divergence of analytic $k$-HMM predictors as a function of order $k$.
   - Identifies the crossover order $k^*$ where the model matches the memory horizon of an optimal $k$-gram Markov predictor.

3. **Lookahead Probability Distributions (Last 100 Tokens Detailed View)**:
   - Full-vocabulary predictive probability distributions across all 181 token bins overlaid with 4 counterfactual physical branches, actual trajectory, and kick markers.

In [1]:
import sys
from pathlib import Path

# Ensure project root and sweeps directory are on sys.path
ROOT = Path.cwd().resolve()
if ROOT.name == "pendulum-sweeps":
    ROOT = ROOT.parents[1]
SWEEPS_DIR = ROOT / "experiments" / "pendulum-sweeps"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(SWEEPS_DIR) not in sys.path:
    sys.path.insert(0, str(SWEEPS_DIR))

import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import numpy as np
import torch

from kl_in_context import (
    load_run,
    run_kl_experiment,
    plot_kl_vs_position,
    plot_kl_vs_k,
    plot_lookahead_probabilities,
    find_crossover_k,
    save_kl_results,
    load_kl_results,
)

print(f"Loaded kl_in_context from {SWEEPS_DIR}")
print(f"CUDA Available: {torch.cuda.is_available()}")

Loaded kl_in_context from /home/manjot/Files/Code/BeliefPhysics/Belief-in-physics/experiments/pendulum-sweeps
CUDA Available: True


## 1. Select Run & Configure Evaluation

In [2]:
# Available run directories
dir_options = []
p_m50 = SWEEPS_DIR / "parallel_output"
p_m200 = SWEEPS_DIR / "parallel_output_m200"
if p_m50.exists():
    dir_options.append(("parallel_output (m=50, seq_len=500) [Ready]", str(p_m50)))
if p_m200.exists():
    m200_ready = len(list((p_m200 / "dt0.2_gamma1_dv1.2_n10").glob("*.pt"))) > 0
    tag = "[Ready]" if m200_ready else "[Training in progress...]"
    dir_options.append((f"parallel_output_m200 (m=200, seq_len=2000) {tag}", str(p_m200)))

dir_dropdown = widgets.Dropdown(
    options=dir_options,
    value=dir_options[0][1] if dir_options else "",
    description="Run Dir:",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="500px"),
)

cfg_dropdown = widgets.Dropdown(
    options=["dt0.2_gamma1_dv1.2_n10"],
    value="dt0.2_gamma1_dv1.2_n10",
    description="Configuration:",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="500px"),
)

width_dropdown = widgets.Dropdown(
    options=[128, 64, 32],
    value=128,
    description="Model Width:",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="250px"),
)

seq_slider = widgets.IntSlider(
    value=500,
    min=100,
    max=1000,
    step=100,
    description="N Sequences:",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="400px"),
)

device_radio = widgets.RadioButtons(
    options=["cuda", "cpu"] if torch.cuda.is_available() else ["cpu"],
    value="cuda" if torch.cuda.is_available() else "cpu",
    description="Device:",
    style={"description_width": "120px"},
)

display(widgets.VBox([
    widgets.HTML("<h4>Experiment Parameters</h4>"),
    dir_dropdown,
    cfg_dropdown,
    widgets.HBox([width_dropdown, device_radio]),
    seq_slider,
]))

## 2. Run Evaluation & Cache Results

In [3]:
out_dir = Path(dir_dropdown.value)
cfg_name = cfg_dropdown.value
w = width_dropdown.value
dev = device_radio.value
n_seq = seq_slider.value
cfg_dir = out_dir / cfg_name
cache_file = cfg_dir / "kl_results.pt"

print(f"Loading models from {cfg_dir} (width={w}, device={dev})...")
run_data = load_run(
    config_name=cfg_name,
    output_dir=out_dir,
    width=w,
    seed=0,
    horizons=("k1", "kn2", "kn"),
    device=dev,
)
loaded_run_data = {"current": run_data}

if cache_file.exists():
    print(f"Loading precomputed KL results from {cache_file}...")
    cached_results = {"latest": load_kl_results(cache_file)}
    print(f"✓ Loaded cached results for {list(cached_results['latest']['horizons'].keys())}.")
else:
    print(f"Computing evaluation on {n_seq} trajectories...")
    res = run_kl_experiment(
        run_data=run_data,
        n_seq=n_seq,
        ks=(0, 1, 2, 3, 4, 5, 6, 8, 10, 15, 20),
        device=dev,
        batch_size=64,
    )
    save_kl_results(res, cache_file)
    cached_results = {"latest": res}
    print(f"✓ Computed and saved to {cache_file}.")

Loading models from /home/manjot/Files/Code/BeliefPhysics/Belief-in-physics/experiments/pendulum-sweeps/parallel_output/dt0.2_gamma1_dv1.2_n10 (width=128, device=cuda)...
Loading precomputed KL results from /home/manjot/Files/Code/BeliefPhysics/Belief-in-physics/experiments/pendulum-sweeps/parallel_output/dt0.2_gamma1_dv1.2_n10/kl_results.pt...
✓ Loaded cached results for ['k1', 'kn2', 'kn'].


## 3. Interactive Plot 1: In-Context KL vs Token Position

Displays model prediction accuracy against **0-HMM** and **1-HMM** baselines across sequence positions. Set `Start Token Pos = 100` to discount the initial phase orbit identification phase.

In [ ]:
plot1_output = widgets.Output()

min_pos_slider = widgets.IntSlider(
    value=100,
    min=0,
    max=300,
    step=10,
    description="Start Token Pos:",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="400px"),
)

scale_radio = widgets.RadioButtons(
    options=[("Both (Linear & Log Subplots)", "both"), ("Log Scale Only", "log"), ("Linear Scale Only", "linear")],
    value="both",
    description="Y-Scale:",
    style={"description_width": "120px"},
)

decomp_checkbox = widgets.Checkbox(
    value=True,
    description="Show Leakage vs In-Subspace Belief Decomposition",
    style={"description_width": "300px"},
)

def update_plot1(*args):
    with plot1_output:
        clear_output(wait=True)
        if "latest" not in cached_results:
            # Attempt auto-load if cache file exists
            cache_file = Path(dir_dropdown.value) / cfg_dropdown.value / "kl_results.pt"
            if cache_file.exists():
                cached_results["latest"] = load_kl_results(cache_file)
            else:
                print("Please run evaluation in Section 2 first.")
                return
        fig = plot_kl_vs_position(
            results=cached_results["latest"],
            yscale=scale_radio.value,
            decompose=decomp_checkbox.value,
            min_pos=min_pos_slider.value,
        )
        display(fig)
        plt.close(fig)

min_pos_slider.observe(update_plot1, names="value")
scale_radio.observe(update_plot1, names="value")
decomp_checkbox.observe(update_plot1, names="value")

display(widgets.VBox([
    widgets.HBox([min_pos_slider, scale_radio]),
    decomp_checkbox,
    plot1_output,
]))

# Render plot immediately on execution
update_plot1()

## 4. Interactive Plot 2: Crossover Markov Order $k^*$

Evaluates average converged KL divergence of analytic $k$-HMM predictors vs Markov order $k$, identifying the crossover order $k^*$ for both Total Model KL and In-Subspace Belief KL.

In [5]:
plot2_output = widgets.Output()

burnin_slider = widgets.IntSlider(
    value=100,
    min=0,
    max=300,
    step=10,
    description="Burn-in Tokens:",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="400px"),
)

def update_plot2(*args):
    with plot2_output:
        clear_output(wait=True)
        if "latest" not in cached_results:
            cache_file = Path(dir_dropdown.value) / cfg_dropdown.value / "kl_results.pt"
            if cache_file.exists():
                cached_results["latest"] = load_kl_results(cache_file)
            else:
                print("Please run evaluation in Section 2 first.")
                return
        fig = plot_kl_vs_k(
            results=cached_results["latest"],
            burn_in_tokens=burnin_slider.value,
        )
        display(fig)
        plt.close(fig)

burnin_slider.observe(update_plot2, names="value")

display(widgets.VBox([
    burnin_slider,
    plot2_output,
]))

# Render plot immediately on execution
update_plot2()

## 5. Interactive Plot 3: Lookahead Predictive Distributions (Last 100 Tokens Detailed View)

Matches `_worker.py` and `lookahead_viz.py`: displays the full model predictive distribution across all 181 token bins as a 2D heatmap over time, overlaid with the 4 counterfactual branches, actual trajectory, and kick markers for the **last 100 tokens** of the sequence.

In [6]:
plot3_output = widgets.Output()

window_slider = widgets.IntSlider(
    value=100,
    min=50,
    max=200,
    step=10,
    description="Window Tokens:",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="400px"),
)

traj_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=15,
    step=1,
    description="Traj Index:",
    style={"description_width": "120px"},
)

def update_plot3(*args):
    with plot3_output:
        clear_output(wait=True)
        if "current" not in loaded_run_data:
            print("Please run evaluation cell above first.")
            return
        fig = plot_lookahead_probabilities(
            run_data=loaded_run_data["current"],
            window_tokens=window_slider.value,
            traj_idx=traj_slider.value,
            seed=42,
            device=device_radio.value,
        )
        display(fig)
        plt.close(fig)

window_slider.observe(update_plot3, names="value")
traj_slider.observe(update_plot3, names="value")

display(widgets.VBox([
    widgets.HBox([window_slider, traj_slider]),
    plot3_output,
]))

update_plot3()